# Classification des clients
Construire un modèle permettant de prédire :_ y = churn_

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split

In [3]:
data = pd.read_csv("data/preprocessed_data.csv")
data.head()

,gender_Female,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,Partner,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,Contract,Churn,tenure,MonthlyCharges,TotalCharges
0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0,1,...,0,0,0,0,1,0,0,0.013889,0.115423,0.003437
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,...,1,0,0,0,0,1,0,0.472222,0.385075,0.217564
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,...,0,0,0,0,1,0,1,0.027778,0.354229,0.012453
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0,...,1,1,0,0,0,1,0,0.625000,0.239303,0.211951
4,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,0,...,0,0,0,0,1,0,1,0.027778,0.521891,0.017462


In [4]:
X = data.drop("Churn", axis=1)
y = data.Churn

display(X.shape)
display(y.shape)

(7043, 24)

(7043,)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=2,
    stratify=y
)

## Tester plusieurs modèles :

In [6]:
# Logistic Regression

lr_model = LogisticRegression(max_iter=100, random_state=1)

params = [
    {
        "penalty": ["l1", "l2"],
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        "solver": ["liblinear"],
        "class_weight": [None, "balanced"]
    },
    {
        "penalty": ["l2"],
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs"],
        "class_weight": [None, "balanced"]
    },
    {
        "penalty": ["elasticnet"],
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        "solver": ["saga"],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "class_weight": [None, "balanced"]
    }
]

scoring = {
    "accuracy": "accuracy",
    "recall": "recall",
    "precision": "precision",
    "balanced_accuracy": "balanced_accuracy",
}

grid = GridSearchCV(
    lr_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall"
)

In [7]:
grid.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.p

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=1),
             param_grid=[{'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'], 'penalty': ['l2'],
                          'solver': ['lbfgs']},
                         {'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'],
                          'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
                          'penalty': ['elasticnet'], 'solver': ['saga']}],
             refit='recall',
             scoring={'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'precision': 'precision', 'recall': 'recall'})

In [8]:
display(grid.best_params_)
display(grid.best_score_)

{'C': 0.001,
 'class_weight': 'balanced',
 'l1_ratio': 0.3,
 'penalty': 'elasticnet',
 'solver': 'saga'}

np.float64(0.8762541806020068)

In [9]:
# Decision Tree
dt_model = DecisionTreeClassifier(random_state=3)

params = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    dt_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall"
)

In [10]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=3),
             param_grid={'class_weight': [None, 'balanced'],
                         'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [3, 5, 7, 10, None],
                         'min_samples_leaf': [1, 2, 5, 10],
                         'min_samples_split': [2, 5, 10, 20]},
             refit='recall',
             scoring={'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'precision': 'precision', 'recall': 'recall'})

In [11]:
display(grid.best_params_)
display(grid.best_score_)

{'class_weight': 'balanced',
 'criterion': 'entropy',
 'max_depth': 5,
 'min_samples_leaf': 5,
 'min_samples_split': 2}

np.float64(0.8006688963210703)

In [12]:
# Random Forest
rf_model = RandomForestClassifier(random_state=4)

params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    rf_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall"
)

In [13]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=4),
             param_grid={'class_weight': [None, 'balanced'],
                         'max_depth': [None, 5, 10, 15, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 5, 10],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             refit='recall',
             scoring={'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'precision': 'precision', 'recall': 'recall'})

In [14]:
display(grid.best_params_)
display(grid.best_score_)

{'class_weight': 'balanced',
 'max_depth': 5,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'min_samples_split': 2,
 'n_estimators': 200}

np.float64(0.7879598662207358)

In [ ]:
# SVM


In [ ]:
# Gradient Boosting ou XGBoost
